# Fusion Formula Tabanli Robot Skorlama Pipeline

Bu notebook iki ayri calisma modu sunar:

- `Full Build`: ilk kez SQLite store ve cluster artefact'larini uretir
- `Rescore From Existing Store`: mevcut `fusion_batch_store.sqlite` korunarak sadece skor katmanini yeniden hesaplar

Tum gorsellestirme hucreleri oncesinde artefact readiness kontrolu zorunludur. Eski-yeni uyumsuzluklarda notebook sessiz fallback yapmaz; yonlendirici hata uretir.


In [ ]:
import copy
from pathlib import Path

CONFIG = {
    "paths": {
        "input_parquet": "data/datathonFINAL.parquet",
        "clean_output_parquet": "data/datathonFINAL_formula_cleaned.parquet",
        "batch_sqlite_db": "data/fusion_batch_store.sqlite",
        "author_scores_parquet": "data/fusion_author_scores.parquet",
        "scored_messages_parquet": "data/fusion_scored_messages.parquet",
        "manifest_path": "data/fusion_manifest.json",
        "author_feature_stage_parquet": "data/fusion_author_feature_stage.parquet",
    },
    "runtime": {
        "mode": "full",
        "use_model_test_sample": False,
        "model_test_rows": 200_000,
        "build_mode": "lean",
        "batch_size": 250_000,
        "max_batches": None,
        "sample_n_rows": None,
        "overwrite_outputs": True,
        "enable_progress_logs": True,
        "progress_every_batches": 10,
        "top_n_domain_context": 128,
        "author_batch_size": 10_000,
        "message_batch_size": 250_000,
    },
    "semantic_adapter": {
        "enabled": True,
        "selected_model_key": "distil_bot_en",
        "models": {
            "distil_bot_en": {
                "model_name": "junaid1993/distilroberta-bot-detection",
                "supported_languages": ["en"],
            },
            "xlmr_base_multilingual": {
                "model_name": "FacebookAI/xlm-roberta-base",
                "supported_languages": "all",
            },
        },
        "unsupported_language_score": 0.50,
        "max_length": 128,
        "batch_size": 64,
        "device": "mps",
    },
    "thresholds": {
        "min_text_len": 5,
        "rare_language_threshold": 100,
        "long_text_start": 30,
        "hourly_penalty_start": 10,
        "hard_hourly_bot_threshold": 15,
        "language_penalty_start": 4,
        "hard_bot_time_window_sec": 300,
        "hard_bot_repeat_threshold": 5,
        "hard_bot_multi_author_threshold": 2,
        "hard_bot_time_cluster_threshold": 3,
        "spam_repeat_threshold": 3,
        "spam_multi_author_threshold": 2,
        "spam_time_cluster_threshold": 3,
    },
    "rules": {
        "long_text_requires_spam": False,
    },
    "derived_thresholds": {},
    "neutral_score_policy": {
        "neutral_score": 0.50,
        "epsilon": 1e-6,
    },
    "dominant_signal_policy": {
        "enabled": True,
        "threshold": 0.68,
        "scope": {
            "message": True,
            "author": True,
            "semantic": False,
        },
        "mode": "floor",
        "repeat_hard_threshold": 5,
        "repeat_hard_target": "final_score",
    },
    "dynamic_final_weighting": {
        "enabled": True,
        "min_confidence_weight": 0.20,
        "power": 2.0,
        "sigmoid_steepness": 8.0,
    },
    "weights": {
        "behavioral_vs_semantic": {
            "behavioral": 0.50,
            "semantic": 0.50,
        },
        "author_vs_message": {
            "author": 0.70,
            "message": 0.30,
        },
        "author_components": {
            "activity": 0.35,
            "timing": 0.25,
            "repetition": 0.26,
            "diversity": 0.14,
            "metadata": 0.00,
        },
        "message_components": {
            "same_text_repeat": 0.20,
            "spam_pattern": 0.60,
            "hashtag_spam": 0.10,
            "token_repetition": 0.04,
            "long_text": 0.02,
            "keyword_signal": 0.04,
        },
    },
}

CONFIG_MODEL_TEST = copy.deepcopy(CONFIG)
CONFIG_MODEL_TEST["paths"].update({
    "input_parquet": "data/fusion_model_test_input_200k.parquet",
    "clean_output_parquet": "data/fusion_model_test_cleaned_200k.parquet",
    "batch_sqlite_db": "data/fusion_model_test_store_200k.sqlite",
    "author_scores_parquet": "data/fusion_model_test_author_scores.parquet",
    "scored_messages_parquet": "data/fusion_model_test_scored_messages.parquet",
    "manifest_path": "data/fusion_model_test_manifest.json",
    "author_feature_stage_parquet": "data/fusion_model_test_author_feature_stage.parquet",
})
MODEL_TEST_ROWS = int(CONFIG["runtime"]["model_test_rows"])
CONFIG_MODEL_TEST["runtime"].update({
    "mode": "model_test_200k_store",
    "batch_size": MODEL_TEST_ROWS,
    "max_batches": 1,
    "sample_n_rows": MODEL_TEST_ROWS,
    "progress_every_batches": 1,
})

RUN_CONFIG = CONFIG_MODEL_TEST if CONFIG["runtime"]["use_model_test_sample"] else CONFIG

CONFIG


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import formula_scoring_pipeline as formula_scoring_pipeline
importlib.reload(formula_scoring_pipeline)

from formula_scoring_pipeline import (
    assert_artifacts_ready,
    build_qa_tables,
    load_manifest,
    plot_hourly_distribution,
    plot_hourly_penalty_curve,
    plot_sentiment_theme_distributions,
    prepare_single_message_input,
    run_formula_pipeline,
    run_formula_pipeline_two_pass,
    run_rescore_from_existing_store,
    score_single_message,
    validate_scored_outputs,
)


def build_in_memory_run_outputs(result: dict, config: dict) -> tuple[dict, dict, dict]:
    scored_df = result["scored_df"]
    author_scores = result["author_scores"]
    clean_df = result["clean_df"]

    result["scored_preview"] = scored_df.head(20).copy()
    result["summary"]["mode"] = config["runtime"]["mode"]
    result["summary"]["semantic_applied_rows"] = int(clean_df["semantic_model_applied_flag"].sum())
    result["paths"] = {
        "input_parquet": config["paths"]["input_parquet"],
        "clean_output_parquet": config["paths"]["clean_output_parquet"],
        "author_scores_parquet": "in-memory",
        "scored_messages_parquet": "in-memory",
        "manifest_json": "in-memory",
    }

    tables = build_qa_tables(result, config)
    tables["derived_thresholds"] = pd.DataFrame({
        "metric": ["hourly_hard_knee"],
        "value": [config.get("derived_thresholds", {}).get("hourly_hard_knee")],
    })
    tables["semantic_adapter_summary"] = pd.DataFrame([{
        "rows": len(clean_df),
        "avg_roberta_score": clean_df["roberta_score"].mean(),
        "min_roberta_score": clean_df["roberta_score"].min(),
        "max_roberta_score": clean_df["roberta_score"].max(),
        "semantic_applied_rows": result["summary"]["semantic_applied_rows"],
    }])
    tables["hourly_hard_authors"] = author_scores.loc[
        author_scores["author_hard_hourly_flag"].eq(1)
    ].sort_values("max_posts_one_hour", ascending=False).head(50)
    tables["score_bands"] = formula_scoring_pipeline.summarize_score_bands(scored_df)
    tables["keyword_validation"] = formula_scoring_pipeline.validate_keyword_signal(scored_df, config)
    tables["hashtag_spam_examples"] = scored_df.sort_values("hashtag_spam_component", ascending=False).head(20)
    tables["token_spam_examples"] = scored_df.sort_values("token_repetition_component", ascending=False).head(20)

    manifest = {
        "pipeline_version": "in-memory-sample",
        "store_schema_version": "in-memory",
        "score_schema_version": "in-memory",
        "created_at": pd.Timestamp.utcnow().isoformat(),
        "last_rescore_at": None,
        "mode": config["runtime"]["mode"],
        "input_parquet": config["paths"]["input_parquet"],
        "rows": len(clean_df),
    }
    tables["manifest_summary"] = pd.DataFrame(
        [{"field": key, "value": value} for key, value in manifest.items()]
    )
    return result, tables, manifest


def ensure_model_test_input(run_config: dict, full_config: dict) -> Path:
    model_test_input_path = Path(run_config["paths"]["input_parquet"])
    model_test_input_path.parent.mkdir(parents=True, exist_ok=True)

    if run_config["runtime"].get("overwrite_outputs", True) or not model_test_input_path.exists():
        source_input_path = Path(full_config["paths"]["input_parquet"])
        first_batch = next(
            pq.ParquetFile(source_input_path).iter_batches(
                batch_size=run_config["runtime"]["sample_n_rows"]
            )
        )
        first_table = pa.Table.from_batches([first_batch])
        pq.write_table(first_table, model_test_input_path, compression="snappy")
    return model_test_input_path


## Input Kontrolu

Veri kaynagini, satir sayisini ve runtime ayarlarini burada kontrol edin.


In [ ]:
RUN_CONFIG = CONFIG_MODEL_TEST if CONFIG["runtime"]["use_model_test_sample"] else CONFIG
input_path = Path(RUN_CONFIG["paths"]["input_parquet"])
source_input_path = Path(CONFIG["paths"]["input_parquet"])

if CONFIG["runtime"]["use_model_test_sample"]:
    planned_rows = RUN_CONFIG["runtime"]["sample_n_rows"]
    inspect_path = input_path if input_path.exists() else source_input_path
else:
    planned_rows = None
    inspect_path = input_path

parquet_file = pq.ParquetFile(inspect_path)

print(f"selected_mode={RUN_CONFIG['runtime']['mode']}")
print(f"input={input_path}")
print(f"inspect_input={inspect_path}")
print(f"inspect_rows={parquet_file.metadata.num_rows:,}")
print(f"selected_rows={planned_rows if planned_rows is not None else parquet_file.metadata.num_rows:,}")
print(f"row_groups={parquet_file.metadata.num_row_groups}")
print(RUN_CONFIG)


## Selected Pipeline Run

`CONFIG["runtime"]["use_model_test_sample"]` true ise 200K sample store build kosar. False ise full batch build kosar.


In [ ]:
RUN_CONFIG = CONFIG_MODEL_TEST if CONFIG["runtime"]["use_model_test_sample"] else CONFIG

if CONFIG["runtime"]["use_model_test_sample"]:
    ensure_model_test_input(RUN_CONFIG, CONFIG)

result = run_formula_pipeline_two_pass(RUN_CONFIG)
tables = result["tables"]
manifest = result.get("manifest")

print("pipeline completed")
print({
    "mode": RUN_CONFIG["runtime"]["mode"],
    "input": RUN_CONFIG["paths"]["input_parquet"],
    "clean_rows": result["summary"].get("clean_rows"),
    "semantic_applied_rows": result["summary"].get("semantic_applied_rows"),
})


## Recompute / Rescore Selected Data

Sample modu açıksa 200K sample store üzerinden rescore çalışır. Full mod açıksa mevcut SQLite store üzerinden rescore çalışır.


In [ ]:
RUN_CONFIG = CONFIG_MODEL_TEST if CONFIG["runtime"]["use_model_test_sample"] else CONFIG

if CONFIG["runtime"]["use_model_test_sample"]:
    ensure_model_test_input(RUN_CONFIG, CONFIG)

result = run_rescore_from_existing_store(RUN_CONFIG)
tables = result["tables"]
manifest = result.get("manifest")
print("rescore completed")
print(result["paths"])


## Artefact Readiness Check

Bu hucre tum gorsellestirme ve tablo hucrelerinden once calismalidir. Burasi manifest, SQLite store ve scored outputs uyumlulugunu siki sekilde dogrular.


In [ ]:
result = assert_artifacts_ready(result, RUN_CONFIG)
tables = result["tables"]
manifest = result["manifest"]

print("artifacts ready")
print({
    "pipeline_version": manifest["pipeline_version"],
    "store_schema_version": manifest["store_schema_version"],
    "score_schema_version": manifest["score_schema_version"],
    "created_at": manifest["created_at"],
    "last_rescore_at": manifest.get("last_rescore_at"),
})


## Manifest Ozeti

In [ ]:
tables["manifest_summary"]

In [ ]:
manifest

## Cleaning ve QA Ozetleri

In [ ]:
tables["summary"]

In [ ]:
tables["derived_thresholds"]

In [ ]:
tables["semantic_adapter_summary"]

In [ ]:
tables["top128_coverage"]

In [ ]:
tables["top_domains"].head(20)

## Author Duzeyi Kontroller

In [ ]:
pd.set_option("display.max_columns", 100)
tables["hourly_heavy_authors"].sort_values("author_score", ascending=False).head(20)

In [ ]:
tables["hourly_hard_authors"].head(20)

In [ ]:
fig = plot_hourly_distribution(result["author_scores"])
fig

In [ ]:
fig = plot_hourly_penalty_curve(result["author_scores"], RUN_CONFIG)
fig

In [ ]:
tables["language_diversity_authors"].head(20)

In [ ]:
fig = plot_sentiment_theme_distributions(result["author_scores"])
fig

## Repeat / Spam / Hard Bot Kontrolleri

In [ ]:
pd.set_option("display.max_columns", 100)
tables["hard_bot_examples"].head(20)

In [ ]:
tables["rapid_fire_examples"].head(20)

In [ ]:
tables["score_bands"]

## Keyword ve Spam Ornekleri

In [ ]:
tables["keyword_validation"]

In [ ]:
tables["hashtag_spam_examples"][[
    "normalized_text",
    "hashtag_count",
    "hashtag_spam_component",
    "message_score",
    "final_score",
]].head(20)

In [ ]:
tables["token_spam_examples"][[
    "normalized_text",
    "max_token_frequency",
    "max_token_ratio",
    "token_repetition_component",
    "message_score",
    "final_score",
]].head(20)

## Skor Ciktilari

In [ ]:
scored_preview = result.get("scored_preview")

if scored_preview is None:
    if "scored_df" in result:
        scored_preview = result["scored_df"].head(20).copy()
    else:
        scored_path = Path(RUN_CONFIG["paths"]["scored_messages_parquet"])
        scored_preview = pd.read_parquet(scored_path).head(20)
    result["scored_preview"] = scored_preview

scored_preview[[
    "author_hash",
    "author_type",
    "normalized_text",
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "final_score_before_rules",
    "final_score",
    "hard_bot_cluster_flag",
    "hard_same_text_repeat_flag",
]].head(20)


## Final Score Dagilimi

Bu hucreler `scored_messages.parquet` icinden dagilimi ve band ayrisimini gosterir. Readiness check bu hucrelerden once calismis olmali.


In [ ]:
score_dist_columns = [
    "final_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "final_score_before_rules",
    "hard_bot_cluster_flag",
    "author_hard_hourly_flag",
]

validate_scored_outputs(RUN_CONFIG, manifest)
score_dist_path = Path(RUN_CONFIG["paths"]["scored_messages_parquet"])
score_dist = pd.read_parquet(score_dist_path, columns=score_dist_columns)

score_dist.describe(include="all")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].hist(score_dist["final_score"], bins=50, color="#2563eb", edgecolor="white")
axes[0].set_title("Final Score Histogram")
axes[0].set_xlabel("final_score")
axes[0].set_ylabel("rows")

axes[1].hist(score_dist["final_score"], bins=50, color="#dc2626", edgecolor="white")
axes[1].set_title("Final Score Histogram (Log Y)")
axes[1].set_xlabel("final_score")
axes[1].set_ylabel("rows")
axes[1].set_yscale("log")

fig.tight_layout()
fig


In [ ]:
score_bands_full = pd.cut(
    score_dist["final_score"],
    bins=[-0.001, 0.4, 0.6, 0.7, 0.85, 0.999999, 1.000001],
    labels=["0.00-0.40", "0.40-0.60", "0.60-0.70", "0.70-0.85", "0.85-<1.0", "1.0"],
    include_lowest=True,
)

score_band_summary = score_bands_full.value_counts(sort=False).rename_axis("band").reset_index(name="rows")
score_band_summary["share"] = score_band_summary["rows"] / max(len(score_dist), 1)
score_band_summary


## Tek Mesaj Inference Scaffold

In [ ]:
single_message = prepare_single_message_input(
    message_text="hi guys good morning how are you today",
    language="en",
    author_hash=None,
    english_keywords="example, test, inference",
    primary_theme="unknown_theme",
)

score_single_message(single_message, result, RUN_CONFIG)

## Dashboard: Score Distribution And Driver Analysis

Bu blok mevcut `scored_messages.parquet`, `fusion_batch_store.sqlite` ve `fusion_author_scores.parquet` dosyalarini kullanir. Full pipeline'i yeniden calistirmaz. Dashboard analizi, genel skor dagilimi ile birlikte dil, platform ve `author_type` kirilimlarini gosterir.

In [ ]:
import sqlite3
import pyarrow.dataset as ds

DASHBOARD_SCORE_COLUMNS = [
    "message_id",
    "author_type",
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "max_token_frequency",
    "max_token_ratio",
    "hard_bot_cluster_flag",
    "hard_same_text_repeat_flag",
    "author_hard_hourly_flag",
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_confidence_weight",
    "roberta_confidence_weight",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "final_score_before_rules",
    "final_score",
]

DASHBOARD_AUTHOR_COLUMNS = [
    "author_hash",
    "activity_risk",
    "timing_risk",
    "repetition_risk",
    "diversity_risk",
    "metadata_risk",
    "max_posts_one_hour",
    "median_interpost_sec",
    "language_nunique",
    "theme_nunique",
    "sentiment_std",
    "author_score",
]

SCORE_BAND_EDGES = [-0.001, 0.4, 0.6, 0.7, 0.85, 0.999999, 1.000001]
SCORE_BAND_LABELS = ["0.00-0.40", "0.40-0.60", "0.60-0.70", "0.70-0.85", "0.85-<1.0", "1.0"]


def build_score_bands(series: pd.Series) -> pd.Categorical:
    return pd.cut(series, bins=SCORE_BAND_EDGES, labels=SCORE_BAND_LABELS, include_lowest=True)


def collapse_top_categories(series: pd.Series, top_n: int = 10, other_label: str = "Other") -> pd.Series:
    base = series.fillna("unknown").astype("string")
    top_values = base.value_counts().head(top_n).index
    collapsed = base.where(base.isin(top_values), other_label)
    return collapsed.astype("category")


def plot_matrix_heatmap(data: pd.DataFrame, title: str, cmap: str = "YlOrRd", value_fmt: str = ".2f"):
    fig, ax = plt.subplots(figsize=(1.5 * max(len(data.columns), 4), 0.8 * max(len(data.index), 4)))
    image = ax.imshow(data.to_numpy(), aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels(data.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index)
    ax.set_title(title)
    for row_idx in range(len(data.index)):
        for col_idx in range(len(data.columns)):
            ax.text(col_idx, row_idx, format(data.iat[row_idx, col_idx], value_fmt), ha="center", va="center", color="black", fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    return fig


def plot_normalized_stack(ax, source_df: pd.DataFrame, group_col: str, top_n: int, title: str):
    grouped = source_df[[group_col, "score_band"]].copy()
    grouped[group_col] = collapse_top_categories(grouped[group_col], top_n=top_n)
    share = pd.crosstab(grouped[group_col], grouped["score_band"], normalize="index")
    share = share.reindex(columns=SCORE_BAND_LABELS, fill_value=0.0)
    share.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel("share")
    ax.legend(title="score_band", bbox_to_anchor=(1.02, 1), loc="upper left")
    return share


validate_scored_outputs(RUN_CONFIG, manifest)

score_path = Path(RUN_CONFIG["paths"]["scored_messages_parquet"])
author_path = Path(RUN_CONFIG["paths"]["author_scores_parquet"])
sqlite_path = Path(RUN_CONFIG["paths"]["batch_sqlite_db"])

score_df = pd.read_parquet(score_path, columns=DASHBOARD_SCORE_COLUMNS)
score_df["message_id"] = pd.to_numeric(score_df["message_id"], errors="coerce").astype("int64")
score_df["author_type"] = score_df["author_type"].fillna("unknown").astype("category")

for column in score_df.columns:
    if column in {"message_id", "author_type"}:
        continue
    if pd.api.types.is_float_dtype(score_df[column]):
        score_df[column] = pd.to_numeric(score_df[column], errors="coerce").astype("float32")
    elif pd.api.types.is_integer_dtype(score_df[column]):
        score_df[column] = pd.to_numeric(score_df[column], errors="coerce", downcast="integer")

with sqlite3.connect(sqlite_path) as conn:
    context_df = pd.read_sql_query(
        """
        SELECT
            message_id,
            COALESCE(NULLIF(language, ''), 'unknown') AS language,
            COALESCE(NULLIF(registered_domain, ''), 'unknown') AS registered_domain,
            keyword_count,
            text_length_chars,
            repeated_token_count_over_2,
            is_long_text_flag
        FROM cleaned_messages
        """,
        conn,
    )

context_df["message_id"] = pd.to_numeric(context_df["message_id"], errors="coerce").astype("int64")
for column in ["language", "registered_domain"]:
    context_df[column] = context_df[column].fillna("unknown").astype("category")
for column in ["keyword_count", "text_length_chars", "repeated_token_count_over_2", "is_long_text_flag"]:
    context_df[column] = pd.to_numeric(context_df[column], errors="coerce", downcast="integer")

dashboard_df = score_df.merge(context_df, on="message_id", how="left", copy=False)
dashboard_df["language_top"] = collapse_top_categories(dashboard_df["language"], top_n=10)
dashboard_df["platform_top"] = collapse_top_categories(dashboard_df["registered_domain"], top_n=10)
dashboard_df["score_band"] = build_score_bands(dashboard_df["final_score"])
dashboard_df["hard_rule_any"] = (
    dashboard_df[["hard_bot_cluster_flag", "hard_same_text_repeat_flag", "author_hard_hourly_flag"]]
    .fillna(0)
    .max(axis=1)
    .astype("int8")
)

author_df = pd.read_parquet(author_path, columns=DASHBOARD_AUTHOR_COLUMNS)
for column in author_df.columns:
    if column == "author_hash":
        continue
    if pd.api.types.is_float_dtype(author_df[column]):
        author_df[column] = pd.to_numeric(author_df[column], errors="coerce").astype("float32")
    elif pd.api.types.is_integer_dtype(author_df[column]):
        author_df[column] = pd.to_numeric(author_df[column], errors="coerce", downcast="integer")

message_component_candidates = [
    "same_text_repeat_component",
    "spam_pattern_component",
    "hashtag_spam_component",
    "token_repetition_component",
    "long_text_component",
    "keyword_signal_component",
]
dashboard_message_component_columns = [col for col in message_component_candidates if col in dashboard_df.columns]
author_component_candidates = ["activity_risk", "timing_risk", "repetition_risk", "diversity_risk", "metadata_risk"]
dashboard_author_component_columns = [col for col in author_component_candidates if col in author_df.columns]

high_risk_dataset = ds.dataset(score_path, format="parquet")
high_risk_columns = [
    "message_id",
    "normalized_text",
    "author_type",
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "final_score",
    "hard_bot_cluster_flag",
    "hard_same_text_repeat_flag",
    "author_hard_hourly_flag",
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "max_token_frequency",
    "max_token_ratio",
]
top_high_risk_ids = (
    dashboard_df.loc[dashboard_df["final_score"].ge(0.85)]
    .sort_values(
        ["final_score", "hard_bot_cluster_flag", "hard_same_text_repeat_flag", "author_hard_hourly_flag"],
        ascending=[False, False, False, False],
    )["message_id"]
    .head(20)
    .tolist()
)
if top_high_risk_ids:
    high_risk_table = high_risk_dataset.to_table(
        columns=high_risk_columns,
        filter=ds.field("message_id").isin(top_high_risk_ids),
    ).to_pandas()
    high_risk_table["message_id"] = pd.to_numeric(high_risk_table["message_id"], errors="coerce").astype("int64")
    high_risk_table = high_risk_table.merge(
        context_df[["message_id", "language", "registered_domain", "keyword_count"]],
        on="message_id",
        how="left",
    )
    high_risk_table = high_risk_table.sort_values(
        ["final_score", "hard_bot_cluster_flag", "hard_same_text_repeat_flag", "author_hard_hourly_flag"],
        ascending=[False, False, False, False],
    )
else:
    high_risk_table = pd.DataFrame(columns=high_risk_columns + ["language", "registered_domain", "keyword_count"])

print({
    "dashboard_rows": int(len(dashboard_df)),
    "author_rows": int(len(author_df)),
    "high_risk_rows": int(len(high_risk_table)),
    "message_component_columns": dashboard_message_component_columns,
    "author_component_columns": dashboard_author_component_columns,
})

In [ ]:
dashboard_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "final_score_mean",
            "final_score_median",
            "final_score_p90",
            "final_score_p99",
            "share_final_score_ge_0_70",
            "share_final_score_ge_0_85",
            "share_final_score_eq_1_0",
            "share_any_hard_rule",
            "avg_behavioral_effective_weight",
            "avg_roberta_effective_weight",
        ],
        "value": [
            len(dashboard_df),
            dashboard_df["final_score"].mean(),
            dashboard_df["final_score"].median(),
            dashboard_df["final_score"].quantile(0.90),
            dashboard_df["final_score"].quantile(0.99),
            dashboard_df["final_score"].ge(0.70).mean(),
            dashboard_df["final_score"].ge(0.85).mean(),
            dashboard_df["final_score"].eq(1.0).mean(),
            dashboard_df["hard_rule_any"].mean(),
            dashboard_df["behavioral_effective_weight"].mean(),
            dashboard_df["roberta_effective_weight"].mean(),
        ],
    }
)
dashboard_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].hist(dashboard_df["final_score"].dropna(), bins=60, color="#1d4ed8", edgecolor="white")
axes[0].set_title("Final Score Histogram")
axes[0].set_xlabel("final_score")
axes[0].set_ylabel("rows")

axes[1].hist(dashboard_df["final_score"].dropna(), bins=60, color="#b91c1c", edgecolor="white")
axes[1].set_title("Final Score Histogram (Log Y)")
axes[1].set_xlabel("final_score")
axes[1].set_ylabel("rows")
axes[1].set_yscale("log")

fig.tight_layout()
fig

In [ ]:
score_view_columns = ["author_score", "message_score", "behavioral_score", "roberta_score", "final_score"]
score_colors = ["#2563eb", "#059669", "#7c3aed", "#d97706", "#dc2626"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.flatten()
for idx, column in enumerate(score_view_columns):
    axes[idx].hist(dashboard_df[column].dropna(), bins=50, color=score_colors[idx], edgecolor="white")
    axes[idx].set_title(column)
    axes[idx].set_xlim(0, 1)
    axes[idx].set_xlabel(column)
    axes[idx].set_ylabel("rows")
axes[-1].axis("off")
fig.tight_layout()
fig

fig, ax = plt.subplots(figsize=(10, 5))
box_data = [dashboard_df[column].dropna().sample(min(150000, dashboard_df[column].notna().sum()), random_state=42) for column in score_view_columns]
ax.boxplot(box_data, labels=score_view_columns, showfliers=False)
ax.set_ylim(0, 1.02)
ax.set_title("Score Comparison Distribution")
ax.set_ylabel("score")
fig.tight_layout()
fig

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

hb = axes[0, 0].hexbin(
    dashboard_df["behavioral_score"],
    dashboard_df["roberta_score"],
    C=dashboard_df["final_score"],
    reduce_C_function=np.mean,
    gridsize=45,
    cmap="viridis",
    mincnt=1,
)
axes[0, 0].set_title("Behavioral Score vs Roberta Score")
axes[0, 0].set_xlabel("behavioral_score")
axes[0, 0].set_ylabel("roberta_score")
fig.colorbar(hb, ax=axes[0, 0], label="avg final_score")

hb2 = axes[0, 1].hexbin(
    dashboard_df["behavioral_effective_weight"],
    dashboard_df["roberta_effective_weight"],
    gridsize=45,
    cmap="magma",
    mincnt=1,
)
axes[0, 1].set_title("Dynamic Fusion Weight Balance")
axes[0, 1].set_xlabel("behavioral_effective_weight")
axes[0, 1].set_ylabel("roberta_effective_weight")
fig.colorbar(hb2, ax=axes[0, 1], label="bin count")

axes[1, 0].hist(dashboard_df["behavioral_effective_weight"].dropna(), bins=50, alpha=0.75, color="#2563eb", edgecolor="white", label="behavioral")
axes[1, 0].hist(dashboard_df["roberta_effective_weight"].dropna(), bins=50, alpha=0.65, color="#dc2626", edgecolor="white", label="roberta")
axes[1, 0].set_title("Effective Weight Distribution")
axes[1, 0].set_xlabel("weight")
axes[1, 0].set_ylabel("rows")
axes[1, 0].legend()

hard_rule_counts = pd.Series(
    {
        "hard_bot_cluster_flag": int(dashboard_df["hard_bot_cluster_flag"].fillna(0).sum()),
        "hard_same_text_repeat_flag": int(dashboard_df["hard_same_text_repeat_flag"].fillna(0).sum()),
        "author_hard_hourly_flag": int(dashboard_df["author_hard_hourly_flag"].fillna(0).sum()),
    }
)
hard_rule_counts.sort_values(ascending=False).plot(kind="bar", ax=axes[1, 1], color=["#991b1b", "#b45309", "#1d4ed8"])
axes[1, 1].set_title("Hard Rule Activations")
axes[1, 1].set_ylabel("rows")
axes[1, 1].tick_params(axis="x", rotation=20)

fig.tight_layout()
fig

In [ ]:
message_driver_columns = [
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "max_token_frequency",
    "max_token_ratio",
    "keyword_count",
    "repeated_token_count_over_2",
    "text_length_chars",
]
message_driver_titles = {
    "same_text_repeat_count": "Same Text Repeat Count",
    "same_text_unique_author_count": "Unique Authors Using Same Text",
    "same_text_time_window_count": "Same Text Time Window Count",
    "hashtag_count": "Hashtag Count",
    "max_token_frequency": "Max Token Frequency",
    "max_token_ratio": "Max Token Ratio",
    "keyword_count": "Keyword Count",
    "repeated_token_count_over_2": "Repeated Token Count >= 3",
    "text_length_chars": "Text Length Chars",
}

available_driver_columns = [column for column in message_driver_columns if column in dashboard_df.columns]
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()
for idx, column in enumerate(available_driver_columns):
    axes[idx].hist(dashboard_df[column].dropna(), bins=50, color="#0f766e", edgecolor="white")
    axes[idx].set_title(message_driver_titles[column])
    axes[idx].set_ylabel("rows")
    axes[idx].set_yscale("log")
for idx in range(len(available_driver_columns), len(axes)):
    axes[idx].axis("off")
fig.tight_layout()
fig

if dashboard_message_component_columns:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
    axes = axes.flatten()
    for idx, column in enumerate(dashboard_message_component_columns):
        axes[idx].hist(dashboard_df[column].dropna(), bins=50, color="#7c3aed", edgecolor="white")
        axes[idx].set_title(column)
        axes[idx].set_xlim(0, 1)
        axes[idx].set_ylabel("rows")
    for idx in range(len(dashboard_message_component_columns), len(axes)):
        axes[idx].axis("off")
    fig.tight_layout()
    fig
else:
    print("message component columns were not persisted in scored_messages.parquet; raw driver distributions are shown instead.")

In [ ]:
author_view_columns = dashboard_author_component_columns or [
    "max_posts_one_hour",
    "median_interpost_sec",
    "language_nunique",
    "theme_nunique",
    "sentiment_std",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
axes = axes.flatten()
for idx, column in enumerate(author_view_columns):
    axes[idx].hist(author_df[column].dropna(), bins=50, color="#2563eb", edgecolor="white")
    axes[idx].set_title(column)
    axes[idx].set_ylabel("authors")
    if column.endswith("_risk") or column == "author_score":
        axes[idx].set_xlim(0, 1)
    if column == "median_interpost_sec":
        axes[idx].set_xscale("log")
for idx in range(len(author_view_columns), len(axes)):
    axes[idx].axis("off")
fig.tight_layout()
fig

In [ ]:
band_metric_columns = [
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "max_token_frequency",
    "max_token_ratio",
    "keyword_count",
]

band_summary = (
    dashboard_df.groupby("score_band", observed=False)[band_metric_columns]
    .mean()
    .reindex(SCORE_BAND_LABELS)
)
band_summary

heatmap_metrics = [
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "keyword_count",
]
plot_matrix_heatmap(band_summary[heatmap_metrics], "Band-Level Mean Metric Heatmap")

high_band = band_summary.loc[["0.00-0.40", "0.70-0.85", "0.85-<1.0", "1.0"], [
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
]].T
high_band.plot(kind="bar", figsize=(12, 5), color=["#94a3b8", "#60a5fa", "#f59e0b", "#dc2626"])
plt.title("Metric Means Across Key Score Bands")
plt.ylabel("mean")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
language_summary = (
    dashboard_df.groupby("language_top", observed=False)
    .agg(rows=("final_score", "size"), mean_final_score=("final_score", "mean"), high_risk_share=("final_score", lambda s: s.ge(0.85).mean()))
    .sort_values("rows", ascending=False)
)
platform_summary = (
    dashboard_df.groupby("platform_top", observed=False)
    .agg(rows=("final_score", "size"), mean_final_score=("final_score", "mean"), high_risk_share=("final_score", lambda s: s.ge(0.85).mean()))
    .sort_values("rows", ascending=False)
)
author_type_summary = (
    dashboard_df.groupby("author_type", observed=False)
    .agg(rows=("final_score", "size"), mean_final_score=("final_score", "mean"), high_risk_share=("final_score", lambda s: s.ge(0.85).mean()))
    .sort_values("rows", ascending=False)
)

print("Language summary")
display(language_summary.head(10))
print("Platform summary")
display(platform_summary.head(10))
print("Author type summary")
display(author_type_summary)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))

language_summary.head(10)["mean_final_score"].sort_values().plot(kind="barh", ax=axes[0, 0], color="#2563eb")
axes[0, 0].set_title("Mean Final Score By Language")
axes[0, 0].set_xlabel("mean_final_score")
plot_normalized_stack(axes[0, 1], dashboard_df, "language", top_n=10, title="Score Band Composition By Language")

platform_summary.head(10)["mean_final_score"].sort_values().plot(kind="barh", ax=axes[1, 0], color="#059669")
axes[1, 0].set_title("Mean Final Score By Platform")
axes[1, 0].set_xlabel("mean_final_score")
plot_normalized_stack(axes[1, 1], dashboard_df, "registered_domain", top_n=10, title="Score Band Composition By Platform")

author_type_summary["mean_final_score"].sort_values().plot(kind="barh", ax=axes[2, 0], color="#7c3aed")
axes[2, 0].set_title("Mean Final Score By Author Type")
axes[2, 0].set_xlabel("mean_final_score")
author_type_band_share = pd.crosstab(dashboard_df["author_type"], dashboard_df["score_band"], normalize="index").reindex(columns=SCORE_BAND_LABELS, fill_value=0.0)
author_type_band_share.plot(kind="bar", stacked=True, ax=axes[2, 1], colormap="plasma")
axes[2, 1].set_title("Score Band Composition By Author Type")
axes[2, 1].set_xlabel("author_type")
axes[2, 1].set_ylabel("share")
axes[2, 1].legend(title="score_band", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
fig

fig, ax = plt.subplots(figsize=(10, 5))
author_type_values = author_type_summary.index.astype(str).tolist()
author_type_box_data = [
    dashboard_df.loc[dashboard_df["author_type"].astype("string") == value, "final_score"].dropna().sample(
        min(120000, int(dashboard_df.loc[dashboard_df["author_type"].astype("string") == value, "final_score"].notna().sum())),
        random_state=42,
    )
    for value in author_type_values
]
ax.boxplot(author_type_box_data, labels=author_type_values, showfliers=False)
ax.set_title("Final Score Distribution By Author Type")
ax.set_ylabel("final_score")
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig

In [ ]:
high_risk_examples = high_risk_table[[
    "normalized_text",
    "language",
    "registered_domain",
    "author_type",
    "author_score",
    "message_score",
    "behavioral_score",
    "roberta_score",
    "final_score",
    "hard_bot_cluster_flag",
    "hard_same_text_repeat_flag",
    "author_hard_hourly_flag",
    "same_text_repeat_count",
    "same_text_unique_author_count",
    "same_text_time_window_count",
    "hashtag_count",
    "max_token_frequency",
    "max_token_ratio",
    "keyword_count",
]].head(20)
high_risk_examples

## Text-Only Fusion Inference

Bu blok final senaryosu icin ayridir. Giris olarak sadece `original_text` kolonu bekleyen bir CSV alir. Author, URL, tarih ve diger metadata alanlari kullanilmaz. Dil label normalize edilmis metin uzerinden uretilir. Skor, `message_score` ve `roberta_score` fusion'i ile verilir.

In [ ]:
import copy
import importlib
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from langdetect import DetectorFactory, detect_langs
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langdetect"])
    from langdetect import DetectorFactory, detect_langs

import formula_scoring_pipeline as formula_scoring_pipeline
importlib.reload(formula_scoring_pipeline)


In [ ]:
TEXT_ONLY_INPUT_CSV = Path("data/YARISMACI_TEST_GIRDISI.csv")
TEXT_ONLY_OUTPUT_CSV = Path("data/final_text_scored.csv")
TEXT_ONLY_DUPLICATE_THRESHOLD = 5
TEXT_ONLY_NEAR_DUPLICATE_THRESHOLD = 0.85
TEXT_ONLY_NEAR_DUPLICATE_MAX_COMPONENT = 0.75

RUN_CONFIG = {
    "semantic_adapter": {
        "enabled": True,
        "selected_model_key": "distil_bot_en",
        "models": {
            "distil_bot_en": {
                "model_name": "junaid1993/distilroberta-bot-detection",
                "supported_languages": ["en"],
            },
            "xlmr_base_multilingual": {
                "model_name": "FacebookAI/xlm-roberta-base",
                "supported_languages": "all",
            },
        },
        "unsupported_language_score": 0.50,
        "max_length": 128,
        "batch_size": 64,
        "device": "mps",
    },
    "thresholds": {
        "min_text_len": 5,
        "long_text_start": 30,
        "hard_bot_time_window_sec": 300,
        "hard_bot_repeat_threshold": 5,
        "hard_bot_multi_author_threshold": 2,
        "hard_bot_time_cluster_threshold": 3,
        "spam_repeat_threshold": 3,
        "spam_multi_author_threshold": 2,
        "spam_time_cluster_threshold": 3,
        "hourly_penalty_start": 10,
        "language_penalty_start": 4,
        "rare_language_threshold": 100,
    },
    "rules": {
        "long_text_requires_spam": False,
    },
    "neutral_score_policy": {
        "neutral_score": 0.50,
        "epsilon": 1e-6,
    },
    "dominant_signal_policy": {
        "enabled": True,
        "threshold": 0.68,
        "scope": {"message": True, "author": True, "semantic": False},
        "mode": "floor",
        "repeat_hard_threshold": 5,
        "repeat_hard_target": "final_score",
    },
    "dynamic_final_weighting": {
        "enabled": True,
        "min_confidence_weight": 0.20,
        "power": 2.0,
        "sigmoid_steepness": 8.0,
    },
    "weights": {
        "behavioral_vs_semantic": {"behavioral": 0.5, "semantic": 0.5},
        "author_vs_message": {"author": 0.7, "message": 0.3},
        "message_components": {
            "same_text_repeat": 0.2,
            "spam_pattern": 0.6,
            "hashtag_spam": 0.1,
            "token_repetition": 0.04,
            "long_text": 0.0,
            "keyword_signal": 0.04,
        },
    },
}


In [ ]:
DetectorFactory.seed = 42
LATIN_CHAR_PATTERN = re.compile(r"[A-Za-z]")


In [ ]:
assert TEXT_ONLY_INPUT_CSV.exists(), f"Missing input CSV: {TEXT_ONLY_INPUT_CSV}"

text_input_df = pd.read_csv(TEXT_ONLY_INPUT_CSV)
required_columns = {"test_id", "text"}
missing_columns = required_columns.difference(text_input_df.columns)
assert not missing_columns, f"CSV must contain columns: {sorted(required_columns)}"

text_input_df = text_input_df[["test_id", "text"]].copy()
text_input_df = text_input_df.rename(columns={"text": "original_text"})
text_input_df["test_id"] = text_input_df["test_id"].astype("string")
text_input_df["input_row_id"] = np.arange(len(text_input_df), dtype="int32")
text_input_df["normalized_text"] = text_input_df["original_text"].map(formula_scoring_pipeline.normalize_text_scalar)
text_input_df[["language", "language_confidence"]] = text_input_df["normalized_text"].apply(
    lambda value: pd.Series(detect_language_from_normalized_text(value))
)
text_input_df["english_keywords"] = pd.NA
text_input_df["primary_theme"] = "unknown_theme"
text_input_df["url"] = pd.NA
text_input_df["date"] = pd.NaT
text_input_df["author_hash"] = pd.NA
text_input_df["sentiment"] = pd.NA
text_input_df["main_emotion"] = pd.NA

text_input_df.head(10)


In [ ]:
text_clean_df = formula_scoring_pipeline.clean_batch(text_input_df.copy(), config=RUN_CONFIG, rare_languages=set())
text_message_features = formula_scoring_pipeline.build_message_features(text_clean_df, RUN_CONFIG)

text_message_features["batch_exact_repeat_count"] = (
    text_message_features.groupby("normalized_text")["normalized_text"].transform("size").fillna(0).astype("int32")
)
exact_component = (((text_message_features["batch_exact_repeat_count"] - 1).clip(lower=0)) / max(TEXT_ONLY_DUPLICATE_THRESHOLD - 1, 1)).clip(0, 1)
text_message_features["exact_repeat_component"] = exact_component.astype("float32")

text_refs = formula_scoring_pipeline.fit_normalization_reference(
    author_features=pd.DataFrame(),
    message_features=text_message_features,
    config=RUN_CONFIG,
)

text_message_scores = formula_scoring_pipeline.compute_message_scores(text_message_features, text_refs, RUN_CONFIG)
text_message_scores["long_text_component"] = 0.0
message_weights = RUN_CONFIG["weights"]["message_components"]
text_message_scores["base_message_score"] = (
    message_weights["same_text_repeat"] * text_message_scores["same_text_repeat_component"]
    + message_weights["spam_pattern"] * text_message_scores["spam_pattern_component"]
    + message_weights["hashtag_spam"] * text_message_scores["hashtag_spam_component"]
    + message_weights["token_repetition"] * text_message_scores["token_repetition_component"]
    + message_weights["keyword_signal"] * text_message_scores["keyword_signal_component"]
).clip(0, 1).astype("float32")
text_message_scores["base_message_score"] = formula_scoring_pipeline.apply_dominant_signal_floor(
    text_message_scores["base_message_score"],
    [
        text_message_scores["same_text_repeat_component"],
        text_message_scores["spam_pattern_component"],
        text_message_scores["hashtag_spam_component"],
        text_message_scores["token_repetition_component"],
        text_message_scores["keyword_signal_component"],
    ],
    RUN_CONFIG,
    scope_key="message",
).astype("float32")
text_message_scores["message_score"] = np.maximum.reduce([
    text_message_scores["base_message_score"].to_numpy(dtype="float32"),
    text_message_scores["exact_repeat_component"].to_numpy(dtype="float32"),
]).astype("float32")
text_message_scores["behavioral_score"] = text_message_scores["message_score"].astype("float32")
text_message_scores["author_score"] = np.nan
text_message_scores["author_hard_hourly_flag"] = 0
text_message_scores["author_type"] = "anonymous"

text_final_df = formula_scoring_pipeline.apply_final_score_weighting(text_message_scores.copy(), RUN_CONFIG)
text_final_core = text_final_df.drop(columns=[
    column for column in ["original_text", "normalized_text", "language"] if column in text_final_df.columns
], errors="ignore")
text_final_df = pd.concat(
    [
        text_input_df[["test_id", "input_row_id", "original_text", "normalized_text", "language", "language_confidence"]].reset_index(drop=True),
        text_final_core.reset_index(drop=True),
    ],
    axis=1,
)

text_final_df[[
    "input_row_id",
    "language",
    "base_message_score",
    "message_score",
    "roberta_score",
    "final_score",
    "batch_exact_repeat_count",
]].head(20)

In [ ]:
text_only_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "english_rows",
            "non_english_rows",
            "unknown_language_rows",
            "final_score_mean",
            "final_score_median",
            "share_final_score_ge_0_70",
            "share_final_score_ge_0_85",
            "share_hashtag_hard_rule",
            "share_exclamation_hard_rule",
            "share_exact_repeat_ge_5",
        ],
        "value": [
            len(text_final_df),
            int(text_final_df["language"].eq("en").sum()),
            int(text_final_df["language"].eq("non_en").sum()),
            int(text_final_df["language"].eq("unknown").sum()),
            text_final_df["final_score"].mean(),
            text_final_df["final_score"].median(),
            text_final_df["final_score"].ge(0.70).mean(),
            text_final_df["final_score"].ge(0.85).mean(),
            text_final_df["hashtag_count"].gt(5).mean(),
            text_final_df["exclamation_count"].gt(10).mean(),
            text_final_df["batch_exact_repeat_count"].ge(TEXT_ONLY_DUPLICATE_THRESHOLD).mean(),
        ],
    }
)
text_only_summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

axes[0].hist(text_final_df["final_score"].dropna(), bins=50, color="#2563eb", edgecolor="white")
axes[0].set_title("Text-Only Final Score Histogram")
axes[0].set_xlabel("final_score")
axes[0].set_ylabel("rows")

language_score_summary = text_final_df.groupby("language", dropna=False)["final_score"].agg(["mean", "count"]).sort_values("count", ascending=False)
language_score_summary["mean"].plot(kind="bar", ax=axes[1], color="#7c3aed")
axes[1].set_title("Mean Final Score By Language Label")
axes[1].set_ylabel("mean_final_score")
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
plt.show()


In [ ]:
text_high_risk_columns = [
    "test_id",
    "input_row_id",
    "original_text",
    "normalized_text",
    "language",
    "hashtag_count",
    "exclamation_count",
    "batch_exact_repeat_count",
    "exact_repeat_component",
    "base_message_score",
    "message_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "final_score",
]

text_high_risk_df = (
    text_final_df.loc[text_final_df["final_score"].ge(0.85), text_high_risk_columns]
    .sort_values(["final_score", "message_score", "roberta_score"], ascending=[False, False, False])
    .head(20)
    .reset_index(drop=True)
)
text_high_risk_df


In [ ]:
text_output_columns = [
    "test_id",
    "input_row_id",
    "original_text",
    "normalized_text",
    "language",
    "language_confidence",
    "hashtag_count",
    "exclamation_count",
    "max_token_frequency",
    "max_token_ratio",
    "repeated_token_count_over_2",
    "batch_exact_repeat_count",
    "exact_repeat_component",
    "base_message_score",
    "message_score",
    "roberta_score",
    "behavioral_effective_weight",
    "roberta_effective_weight",
    "final_score",
]

text_final_df[text_output_columns].to_csv(TEXT_ONLY_OUTPUT_CSV, index=False)
print(f"text-only scored output written to {TEXT_ONLY_OUTPUT_CSV}")
